# PHARVO-beta: Low Stock Display & Threshold Verification Test

**Objective:** Verify that an authorized user (`rafi`) can observe low-stock alerts on the **Dashboard** and view filtered low-stock medicines in the **Medicines & Inventory** module, asserting that each item's stock quantity is below or at its reorder threshold.

### Test Steps
1. Sign in with staff credentials (`rafi`).
2. Assert presence of the **Low Stock Panel** on the Dashboard.
3. Navigate to **Medicines & Inventory** and click the **Low Stock** filter pill.
4. Verify low-stock badges and assert threshold (`stock_quantity <= reorder_level`).

### Prerequisites
```bash
pip install selenium webdriver-manager
```
Ensure PHARVO frontend is running at `http://localhost:5173` and backend at `http://localhost:8000`.

In [ ]:
import time
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# --- Configuration & Test Credentials ---
BASE_URL = "http://localhost:5173"
USERNAME = "rafi"
PASSWORD = "password"  # Replace with actual password

# Step 1: Open browser and maximize window
driver = webdriver.Chrome()
driver.maximize_window()

# Set explicit wait helper (up to 10 seconds)
wait = WebDriverWait(driver, 10)

try:
    print("[INFO] Starting Low Stock Display Test...")

    # Step 2: Open login page and sign in
    driver.get(f"{BASE_URL}/")

    username_field = wait.until(
        EC.visibility_of_element_located((By.ID, "username"))
    )
    password_field = wait.until(
        EC.visibility_of_element_located((By.ID, "password"))
    )

    username_field.clear()
    username_field.send_keys(USERNAME)

    password_field.clear()
    password_field.send_keys(PASSWORD)

    sign_in_button = wait.until(
        EC.element_to_be_clickable((By.ID, "sign-in-btn"))
    )
    sign_in_button.click()

    # =========================================================================
    # PART A: VERIFY LOW STOCK PANEL ON DASHBOARD
    # =========================================================================
    print("[INFO] Checking Low Stock panel on Dashboard...")

    # Wait for Dashboard header
    wait.until(
        EC.visibility_of_element_located((By.XPATH, "//header//h1[contains(text(), 'Dashboard')]"))
    )

    # Locate the Low Stock panel on Dashboard
    low_stock_panel = wait.until(
        EC.visibility_of_element_located(
            (By.XPATH, "//*[contains(text(), 'Low Stock') and (contains(., 'reorder') or contains(., 'threshold'))]/ancestor::section | //*[contains(text(), 'Low Stock')]/ancestor::div[contains(@class, 'staff-card')]")
        )
    )
    print("PASS: Low Stock panel located on Dashboard.")

    # Check if there are listed low-stock items or empty state on Dashboard
    dashboard_items = low_stock_panel.find_elements(By.XPATH, ".//li")
    if dashboard_items:
        print(f"[INFO] Dashboard shows {len(dashboard_items)} priority low-stock item(s).")
        for i, item in enumerate(dashboard_items[:3], 1):
            print(f"      {i}. {item.text.replace(chr(10), ' | ')}")
    else:
        print("[INFO] Dashboard Low Stock panel displays no critical items or empty state.")

    # =========================================================================
    # PART B: VERIFY LOW STOCK FILTER IN MEDICINES & INVENTORY
    # =========================================================================
    print("[INFO] Navigating to Medicines & Inventory...")

    medicines_nav = wait.until(
        EC.element_to_be_clickable(
            (By.XPATH, "//aside//button[contains(., 'Medicines & Inventory')]")
        )
    )
    medicines_nav.click()

    wait.until(
        EC.visibility_of_element_located(
            (By.XPATH, "//header//h1[contains(text(), 'Medicines & Inventory')]")
        )
    )

    # Click the 'Low Stock' filter pill in the status group
    low_stock_pill = wait.until(
        EC.element_to_be_clickable(
            (By.XPATH, "//div[@role='group' and @aria-label='Filter by stock status']//button[contains(., 'Low Stock')]")
        )
    )
    low_stock_pill.click()
    print("[INFO] Clicked 'Low Stock' filter pill.")

    time.sleep(1)  # Allow filter to update table

    # Collect visible low-stock rows from inventory table
    low_stock_rows = driver.find_elements(
        By.XPATH, "//table[contains(@class, 'med-table')]//tbody//tr[contains(@class, 'med-row-tr')]"
    )

    if low_stock_rows:
        print(f"PASS: Located {len(low_stock_rows)} low-stock item(s) in inventory table.")

        # =====================================================================
        # PART C: ASSERT THRESHOLD (Stock Quantity <= Reorder Level)
        # =====================================================================
        verified_count = 0
        for row in low_stock_rows:
            med_name = row.find_element(By.XPATH, ".//td[1]//span[1]").text
            stock_text = row.find_element(By.XPATH, ".//td[5]").text
            reorder_text = row.find_element(By.XPATH, ".//td[6]").text
            status_badge = row.find_element(By.XPATH, ".//td[8]").text

            print(f"      - {med_name}: Stock = [{stock_text}], Reorder Level = [{reorder_text}], Status = [{status_badge}]")

            # Extract numeric value for stock and reorder if available
            try:
                # Strip out unit names (e.g. '10 PC' -> 10)
                stock_num = int(''.join(filter(str.isdigit, stock_text.split()[0])))
                reorder_num = int(''.join(filter(str.isdigit, reorder_text)))
                assert stock_num <= reorder_num, f"Assertion failed: Stock ({stock_num}) > Reorder ({reorder_num})"
                verified_count += 1
            except (ValueError, IndexError):
                # If pack breakdown is complex, verify the status badge confirms Low Stock
                assert "Low" in status_badge or "Out" in status_badge, f"Unexpected badge: '{status_badge}'"
                verified_count += 1

        print(f"PASS: Threshold verified on {verified_count} low-stock item(s) (Stock <= Reorder Level).")

    else:
        # If no medicines are currently low stock, verify empty state is cleanly shown
        empty_msg = driver.find_element(By.XPATH, "//*[contains(text(), 'No medicines found') or contains(text(), 'Inventory is above')]")
        print(f"PASS: No items currently below reorder level. Clean state displayed: '{empty_msg.text}'.")

except Exception as error:
    print(f"FAIL: Low Stock Display test encountered error: {error}")

finally:
    # Step 4: Close browser
    print("[INFO] Cleaning up and closing browser...")
    driver.quit()
